# Лабораторная работа №4, вариант 3

Решение квадратной СЛАУ $Ax=b$ методом Гаусса с выбором главного элемента
по столбцу. Тип данных — `double`, $n\leq10000$.

В notebook сохранены результаты CPU и GPU, сравнение десяти
конфигураций и профиль `nvprof`. Источники — CSV и журналы в `report/results/`.

## Метод и размещение данных

На шаге $k$ выбирается строка

$$p=\operatorname*{arg\,max}_{i=k,\ldots,n-1}|a_{ik}|.$$

Строки $k$ и $p$ меняются местами в ещё используемой части расширенной
матрицы $[A\mid b]$. Для $i>k$ вычисляются множители
$l_i=a_{ik}/a_{kk}$, затем параллельно обновляются элементы:

$$a_{ij}\leftarrow a_{ij}-l_i a_{kj},\qquad i>k,\quad j=k+1,\ldots,n.$$

Последний столбец $j=n$ — правая часть. Множители хранятся в столбце $k$;
ядро обновления не записывает в этот столбец, поэтому другие потоки читают
неизменные значения. После прямого хода последовательно по $k=n-1,\ldots,0$
находим $x_k=b_k/a_{kk}$ и параллельно обновляем $b_i\leftarrow b_i-a_{ik}x_k$
для $i<k$. Запуски в одном CUDA stream обеспечивают порядок шагов.

## Объединение обращений к глобальной памяти

Расширенная матрица хранится **по столбцам**:

```cpp
matrix[column * n + row]
```

В двумерном ядре `eliminate` координата `x` задаёт строку, а `y` — столбец.
Соседние `threadIdx.x` читают и записывают соседние `double` одного столбца.
При `block.x = 32` warp обрабатывает один столбец; при меньшем `block.x`
warp захватывает несколько непрерывных участков разных столбцов. Начало
активной части `k + 1` не всегда выровнено, поэтому число транзакций следует
проверять профилировщиком. Два цикла с шагом всей сетки позволяют обрабатывать
матрицы, превышающие размер сетки потоков.

Поиск максимума и вычисление множителей также читают непрерывные участки
столбцов. Перестановка двух строк обращается к памяти с шагом `n`, поэтому
полной коалесценции для неё нет; суммарно это $O(n^2)$ операций против
$O(n^3)$ обновлений в прямом ходе.

Число арифметических операций — $O(n^3)$ (ведущий член примерно
$2n^3/3$), память — $O(n^2)$. При $n=10000$ расширенная матрица занимает
`10000 × 10001 × 8 = 800 080 000` байт; вместе с вектором решения явные
буферы GPU занимают `800 160 000` байт, примерно **763,1 МиБ**. Дополнительно
нужны временная память Thrust, CUDA-контекст и память CPU. Поэтому максимальный
размер из условия не включён в короткую серию по умолчанию.


## Точность и область применимости

Решается невырожденная квадратная СЛАУ. Главный элемент сравнивается с точным
нулём: порог $10^{-7}$ из задания относится к другим вариантам и здесь
отсекал бы допустимые системы с малыми коэффициентами. Нулевой либо
нечисловой главный элемент приводит к сообщению об ошибке.

Вывод `%.10e` содержит 11 значащих цифр. Это обеспечивает требуемую точность
представления $10^{-10}$; точность самого решения дополнительно зависит от
обусловленности матрицы. Выбор главного элемента улучшает устойчивость,
но не гарантирует малую относительную ошибку на любой плохо обусловленной
системе.

В экспериментах известен вектор $x^*$, поэтому проверяются обе величины:

$$\varepsilon_x=\frac{\lVert x-x^*\rVert_\infty}{\lVert x^*\rVert_\infty},\qquad
\eta=\frac{\lVert Ax-b\rVert_\infty}
{\lVert A\rVert_\infty\lVert x\rVert_\infty+\lVert b\rVert_\infty}.$$

## Подготовка

Запускайте notebook из корня репозитория или папки `lab4-variant3`.
Нужны установленные Python, CUDA Toolkit и `g++`; для новых GPU-замеров —
доступное CUDA-устройство. Готовые результаты можно просмотреть и без GPU.

Ячейка ниже находит папку лабораторной, настраивает временные каталоги
внутри проекта и определяет функции запуска команд и вывода таблиц.
Пакеты и данные автоматически не скачиваются.

In [1]:
import csv
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime, timezone, timedelta
from html import escape
from pathlib import Path

try:
    from IPython.display import display
    rich_display = True
except ImportError:
    rich_display = False

current = Path.cwd().resolve()
candidates = [current, current / 'lab4-variant3']
lab = next((p for p in candidates if (p / 'lab4.cu').is_file()
            and (p / 'report' / 'run_benchmarks.py').is_file()), None)
if lab is None:
    raise RuntimeError('Запустите notebook из корня репозитория или lab4-variant3.')
work = lab / 'work'
results = lab / 'report' / 'results'
work.mkdir(exist_ok=True)
results.mkdir(exist_ok=True)
for name, directory in {
    'TMPDIR': work / 'tmp',
    'CUDA_CACHE_PATH': work / 'cuda-cache',
    'NVPROF_TMPDIR': work / 'tmp',
    'XDG_CACHE_HOME': work / 'cache',
    'IPYTHONDIR': work / 'ipython',
    'JUPYTER_CONFIG_DIR': work / 'jupyter',
    'JUPYTER_DATA_DIR': work / 'jupyter-data',
    'JUPYTER_RUNTIME_DIR': work / 'jupyter-runtime',
}.items():
    directory.mkdir(exist_ok=True)
    os.environ[name] = str(directory)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

def run(args, *, check=True, input=None):
    process = subprocess.run([str(a) for a in args], cwd=lab,
                             capture_output=True, text=True, input=input)
    if check and process.returncode != 0:
        raise RuntimeError(process.stderr.strip() or process.stdout.strip()
                           or f'Код возврата: {process.returncode}')
    return process

def show_table(headers, rows):
    headers = [str(value) for value in headers]
    rows = [[str(value) for value in row] for row in rows]
    all_rows = [headers, *rows]
    widths = [max(len(row[i]) for row in all_rows) for i in range(len(headers))]
    plain = '\n'.join(' | '.join(value.ljust(widths[i]) for i, value in enumerate(row))
                      for row in all_rows)
    if rich_display:
        head = '<tr>' + ''.join(f'<th>{escape(h)}</th>' for h in headers) + '</tr>'
        body = ''.join('<tr>' + ''.join(f'<td>{escape(value)}</td>' for value in row)
                       + '</tr>' for row in rows)
        display({'text/html': '<table>' + head + body + '</table>',
                 'text/plain': plain}, raw=True)
    else:
        print(plain)

print('Лабораторная:', lab)

Лабораторная: /home/denis/Desktop/VMK/cuda-course/lab4-variant3


## План эксперимента

По умолчанию используются размеры **32, 256, 1024**, 100 прогревочных запусков и
**пять измеряемых повторений** на CPU и для каждой из десяти конфигураций GPU.
Изменяемые сетка и блок относятся к двумерному ядру `eliminate`; размеры
одномерных вспомогательных ядер остаются фиксированными.

Матрица воспроизводима: LCG с seed 7, исходная диагональ
$2+\sum_{j\ne i}|a_{ij}|$, затем порядок строк обращается, чтобы требовались
перестановки. Элементы вне диагонали лежат в $[-0.5/n,0.5/n)$.
Известное решение — $x_i^*=(-1)^i(1+(i\bmod7)/8)$, правая часть вычисляется
из сохранённых `double` с накоплением в `long double`. Для каждого повтора
восстанавливается одна и та же исходная матрица.

Время — среднее **полного решения системы**, измеренное `std::chrono`:
GPU синхронизируется до и после вызова. Включены поиск главного элемента,
Thrust (включая его внутренние выделения памяти), запуски ядер и внутренние
синхронизации. Исключены выделение внешних буферов матрицы и решения,
копирование исходной матрицы, получение готового вектора и проверки.
Это не время только одного ядра и не полное время программы с вводом/выводом.
Сумма времён ядер в `nvprof` может отличаться от этих значений.


In [2]:
SIZES = [32, 256, 1024]
REPEATS = 5
WARMUP = 100
PROFILE_SIZE = 256
AUTO_RUN_GPU = True   
RUN_GPU = False       
RUN_PROFILE = False  
RERUN_CPU = False    

CONFIGS = [
    (1, 1, 8, 8), (4, 4, 8, 8), (8, 8, 16, 8), (16, 16, 16, 8),
    (16, 16, 32, 8), (32, 32, 32, 8), (16, 16, 16, 16),
    (32, 32, 16, 16), (16, 16, 32, 16), (32, 32, 32, 16),
]
show_table(['№', 'Сетка (x × y)', 'Блок (x × y)', 'Потоков в блоке'],
           [[i, f'{gx} × {gy}', f'{bx} × {by}', bx * by]
            for i, (gx, gy, bx, by) in enumerate(CONFIGS, 1)])

№,Сетка (x × y),Блок (x × y),Потоков в блоке
1,1 × 1,8 × 8,64
2,4 × 4,8 × 8,64
3,8 × 8,16 × 8,128
4,16 × 16,16 × 8,128
5,16 × 16,32 × 8,256
6,32 × 32,32 × 8,256
7,16 × 16,16 × 16,256
8,32 × 32,16 × 16,256
9,16 × 16,32 × 16,512
10,32 × 32,32 × 16,512


## Запуск и источники результатов

Сохранённые результаты читаются без повторных измерений. Если GPU-замеров
не хватает и GPU доступен текущему процессу, серия и профиль запускаются
автоматически. Для принудительного повторения установите `RUN_GPU = True`
и/или `RUN_PROFILE = True` выше.

Если профилирование завершилось ошибкой после успешных измерений, проверенные
CSV всё равно используются. Профиль и время решения системы проверяются отдельно.

In [3]:
def read_metadata(mode):
    path = results / f'metadata-{mode}.json'
    return json.loads(path.read_text()) if path.is_file() else {}

def source_is_current(mode, size):
    metadata = read_metadata(mode)
    arguments = metadata.get('arguments', {})
    filename = f'benchmark-{size}.csv' if mode == 'gpu' else f'cpu-{size}.csv'
    return ((metadata.get('measurements_status') == 'complete'
             or metadata.get('status') == 'complete')
            and filename in metadata.get('result_files', [])
            and arguments.get('repeats') == REPEATS
            and arguments.get('warmup') == WARMUP
            and (results / filename).is_file())

base_command = [sys.executable, '-B', lab / 'report' / 'run_benchmarks.py',
                '--sizes', *map(str, SIZES), '--repeats', str(REPEATS),
                '--warmup', str(WARMUP)]
missing_gpu = [n for n in SIZES if not source_is_current('gpu', n)]
nvidia_smi = shutil.which('nvidia-smi')
probe = run([nvidia_smi, '--query-gpu=name', '--format=csv,noheader'], check=False) if nvidia_smi else None
gpu_available = probe is not None and probe.returncode == 0
run_automatically = AUTO_RUN_GPU and missing_gpu and gpu_available
if RUN_GPU or RUN_PROFILE or run_automatically:
    profile = RUN_PROFILE or bool(run_automatically)
    command = base_command + (['--profile', '--profile-size', str(PROFILE_SIZE)] if profile else [])
    process = run(command, check=False)
    if process.returncode:
        print('Запуск завершился с ошибкой; проверяем, какие результаты сохранены:')
        print((process.stderr.strip() or process.stdout.strip())[-2500:])
    else:
        print('Серия CPU/GPU' + (' и профиль выполнены.' if profile else ' выполнена.'))
elif missing_gpu and not gpu_available:
    print('GPU недоступен текущему процессу. Используются готовые измерения CPU.')

missing_sizes = [n for n in SIZES if not source_is_current('cpu', n)
                 and not source_is_current('gpu', n)]
if RERUN_CPU or missing_sizes:
    selected = SIZES
    command = [sys.executable, '-B', lab / 'report' / 'run_benchmarks.py', '--cpu-only',
               '--sizes', *map(str, selected), '--repeats', str(REPEATS),
               '--warmup', str(WARMUP)]
    process = run(command)
    print(process.stdout.strip()[-2000:] or 'Серия CPU выполнена.')
else:
    print('Используются сохранённые CSV; новые измерения не запускались.')

if read_metadata('gpu').get('status') == 'failed':
    print('Последний GPU-запуск завершился с ошибкой; CSV принимаются только после завершённых измерений:')
    print(str(read_metadata('gpu').get('error', 'Причина указана в metadata-gpu.json'))[:2000])

Используются сохранённые CSV; новые измерения не запускались.


## Проверка сохранённых данных

Одна строка GPU соответствует одному размеру и одной конфигурации; для
CPU-only используется одна строка на размер. До сравнения проверяются
параметры серии, полнота конфигураций, число проверок, конечность значений
и допуски точности. Таблицы используют CPU-время из того же GPU-запуска,
чтобы не смешивать независимые серии при вычислении ускорения.


In [4]:
def load_rows(path, size, mode):
    with path.open(newline='') as file:
        rows = list(csv.DictReader(file))
    assert len(rows) == (len(CONFIGS) if mode == 'gpu' else 1), f'Неполный CSV: {path.name}'
    for row in rows:
        assert row['mode'] == mode and int(row['n']) == size, path.name
        assert int(row['repeats']) == REPEATS and int(row['warmup']) == WARMUP, (
            f'Параметры {path.name} отличаются: измените SIZES/REPEATS/WARMUP или перезапустите серию.')
        assert int(row['errors']) == 0, f'Ошибка корректности: {path.name}'
        assert int(row['cpu_checks']) == REPEATS + WARMUP, path.name
        for prefix in (['cpu', 'gpu'] if mode == 'gpu' else ['cpu']):
            milliseconds = float(row[f'{prefix}_ms'])
            relative = float(row[f'{prefix}_rel_error'])
            residual = float(row[f'{prefix}_scaled_residual'])
            assert math.isfinite(milliseconds) and milliseconds > 0, path.name
            assert math.isfinite(relative) and 0 <= relative <= 1e-10, path.name
            assert math.isfinite(residual) and 0 <= residual <= 1e-10, path.name
        if mode == 'gpu':
            assert int(row['gpu_checks']) == REPEATS + WARMUP, path.name
            ratio = float(row['cpu_ms']) / float(row['gpu_ms'])
            assert math.isclose(float(row['speedup']), ratio, rel_tol=1e-4, abs_tol=1e-6), path.name
        else:
            assert int(row['gpu_checks']) == 0 and not row['gpu_ms'] and not row['speedup'], path.name
    if mode == 'gpu':
        actual = [(int(r['grid_x']), int(r['grid_y']), int(r['block_x']), int(r['block_y'])) for r in rows]
        assert actual == CONFIGS, f'Другой набор конфигураций: {path.name}'
    return rows

cpu_measurements = []
gpu_measurements = []
source_files = []
for size in SIZES:
    gpu_path = results / f'benchmark-{size}.csv'
    cpu_path = results / f'cpu-{size}.csv'
    if source_is_current('gpu', size):
        rows = load_rows(gpu_path, size, 'gpu')
        gpu_measurements.extend(rows)
        cpu_measurements.append(rows[0])
        source_files.append(gpu_path)
    elif source_is_current('cpu', size):
        rows = load_rows(cpu_path, size, 'cpu')
        cpu_measurements.extend(rows)
        source_files.append(cpu_path)
    else:
        raise FileNotFoundError(f'Нет результатов для n={size}.')

print(f'Проверено: {len(cpu_measurements)} размеров CPU, {len(gpu_measurements)} строк GPU.')

Проверено: 3 размеров CPU, 30 строк GPU.


## Характеристики среды

Следующая таблица описывает **текущий** компьютер. История сохранённых
запусков содержится в `metadata-cpu.json` и `metadata-gpu.json`; при переносе
CSV на другую машину опираться нужно на эти файлы. Время изменения CSV
выше не подменяет время выполнения эксперимента.


In [5]:
def command_info(command):
    if not command or not shutil.which(str(command[0])):
        return 'Команда отсутствует'
    process = run(command, check=False)
    text = (process.stdout.strip() if process.returncode == 0
            else (process.stderr.strip() or process.stdout.strip()))
    return text[:1200] or f'Код возврата: {process.returncode}'

def short_version(text):
    lines = text.strip().splitlines()
    return next((line for line in lines if 'Cuda compilation tools, release' in line),
                lines[0] if lines else 'Нет данных')

cpu_path = Path('/proc/cpuinfo')
cpu_name = next((line.split(':', 1)[1].strip() for line in cpu_path.read_text().splitlines()
                 if line.startswith('model name')), platform.processor()) if cpu_path.exists() else platform.processor()
mem_path = Path('/proc/meminfo')
ram = next((line.split()[1] for line in mem_path.read_text().splitlines()
            if line.startswith('MemTotal:')), None) if mem_path.exists() else None
nvcc = shutil.which('nvcc')
if not nvcc:
    candidates = [Path('/usr/local/cuda/bin/nvcc'), Path('/usr/local/cuda-12.8/bin/nvcc')]
    nvcc = next((str(p) for p in candidates if p.is_file()), None)
current_machine = [
    ['CPU', cpu_name],
    ['Логических потоков', os.cpu_count()],
    ['ОЗУ', f'{int(ram) / 2**20:.2f} ГиБ' if ram else 'Нет данных'],
    ['ОС', f'{platform.system()} {platform.release()} {platform.machine()}'],
    ['Python', platform.python_version()],
    ['g++', command_info(['g++', '--version']).splitlines()[0]],
    ['CUDA Toolkit', short_version(command_info([nvcc, '--version'])) if nvcc else 'nvcc не найден в PATH'],
    ['GPU / драйвер', command_info(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                                   '--format=csv,noheader'])],
]
show_table(['Параметр текущей среды', 'Значение'], current_machine)
device_info = work / 'device_info'
if device_info.is_file():
    process = run([device_info], check=False)
    if process.returncode == 0:
        rows = list(csv.reader(process.stdout.splitlines(), delimiter='\t'))
        show_table(['Параметр CUDA устройства', 'Значение'], [r[:2] for r in rows if len(r) >= 2])
    else:
        print('CUDA Runtime:', (process.stderr.strip() or process.stdout.strip())[:1000])
else:
    print('Характеристики CUDA Runtime появятся после успешного запуска GPU-стенда.')

for mode in ['cpu', 'gpu']:
    metadata = read_metadata(mode)
    if metadata:
        print(f'Сохранённый запуск: metadata-{mode}.json')
        arguments = metadata.get('arguments', {})
        summary = [
            ['Статус', metadata.get('status', 'Нет данных')],
            ['Начало (UTC)', metadata.get('started_utc', 'Нет данных')],
            ['Завершение (UTC)', metadata.get('finished_utc', 'Нет данных')],
            ['Размеры / повторы / прогрев', f"{arguments.get('sizes')} / {arguments.get('repeats')} / {arguments.get('warmup')}"],
            ['CPU запуска', metadata.get('cpu_model', 'Нет данных')],
            ['Компилятор CPU', metadata.get('cxx_version', 'Нет данных').splitlines()[0]],
            ['Файлы результата', ', '.join(metadata.get('result_files', [])) or 'Нет'],
        ]
        if mode == 'gpu':
            summary += [['Профилирование', metadata.get('profile_status', 'Не запрашивалось')],
                        ['CUDA Toolkit', short_version(metadata.get('nvcc_version', 'Нет данных'))]]
        show_table(['Параметр серии', 'Значение'], summary)
        if metadata.get('error'):
            print('Причина ошибки:', str(metadata['error'])[:2000])


Параметр текущей среды,Значение
CPU,Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Логических потоков,12
ОЗУ,15.50 ГиБ
ОС,Linux 6.8.0-138-generic x86_64
Python,3.10.12
g++,g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
CUDA Toolkit,"Cuda compilation tools, release 12.8, V12.8.93"
GPU / драйвер,"NVIDIA GeForce GTX 1050, 570.211.01, 3072 MiB"


Параметр CUDA устройства,Значение
Модель,NVIDIA GeForce GTX 1050
Compute capability,6.1
CUDA runtime,12080
CUDA driver,12080
"Глобальная память, байт",3158114304
"Разделяемая память на блок, байт",49152
"Константная память, байт",65536
Регистров на блок,65536
Максимум потоков на блок,1024
"Размеры блока (x, y, z)",1024 x 1024 x 64


Сохранённый запуск: metadata-cpu.json


Параметр серии,Значение
Статус,complete
Начало (UTC),2026-11-08T13:04:55.589701+00:00
Завершение (UTC),2026-11-08T13:05:34.183929+00:00
Размеры / повторы / прогрев,"[32, 256, 1024] / 5 / 100"
CPU запуска,Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Компилятор CPU,g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Файлы результата,"cpu-32.csv, cpu-256.csv, cpu-1024.csv"


Сохранённый запуск: metadata-gpu.json


Параметр серии,Значение
Статус,complete
Начало (UTC),2026-11-08T13:04:54.523419+00:00
Завершение (UTC),2026-11-08T13:11:15.556086+00:00
Размеры / повторы / прогрев,"[32, 256, 1024] / 5 / 100"
CPU запуска,Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz
Компилятор CPU,g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Файлы результата,"benchmark-32.csv, benchmark-256.csv, benchmark-1024.csv"
Профилирование,complete
CUDA Toolkit,"Cuda compilation tools, release 12.8, V12.8.93"


## Измеренные времена и точность

Время указано в миллисекундах и усреднено по повторам из параметров серии.
Ошибки и невязки — максимумы по всем проверенным запускам, включая прогрев.
CPU и GPU сравниваются по данным одной серии.

In [6]:
show_table(['n', 'CPU, мс', 'Проверок CPU', 'Ошибка x', 'Масштаб. невязка'],
           [[r['n'], f"{float(r['cpu_ms']):.6f}", r['cpu_checks'],
             f"{float(r['cpu_rel_error']):.3e}", f"{float(r['cpu_scaled_residual']):.3e}"]
            for r in cpu_measurements])
if gpu_measurements:
    show_table(['n', 'Сетка', 'Блок', 'CPU, мс', 'GPU, мс', 'CPU / GPU', 'Ошибка x (GPU)', 'Невязка (GPU)'],
               [[r['n'], f"{r['grid_x']} × {r['grid_y']}", f"{r['block_x']} × {r['block_y']}",
                 f"{float(r['cpu_ms']):.6f}", f"{float(r['gpu_ms']):.6f}",
                 f"{float(r['cpu_ms']) / float(r['gpu_ms']):.2f}",
                 f"{float(r['gpu_rel_error']):.3e}", f"{float(r['gpu_scaled_residual']):.3e}"]
                for r in gpu_measurements])
else:
    print('Таблица CPU/GPU будет заполнена после RUN_GPU = True на доступном CUDA GPU.')


n,"CPU, мс",Проверок CPU,Ошибка x,Масштаб. невязка
32,0.007269,105,5.075e-16,2.356e-16
256,3.280673,105,2.030e-15,9.480e-16
1024,247.697101,105,5.837e-15,2.737e-15


n,Сетка,Блок,"CPU, мс","GPU, мс",CPU / GPU,Ошибка x (GPU),Невязка (GPU)
32,1 × 1,8 × 8,0.007269,1.352835,0.01,5.075e-16,2.377e-16
32,4 × 4,8 × 8,0.007269,1.069679,0.01,5.075e-16,2.377e-16
32,8 × 8,16 × 8,0.007269,1.236442,0.01,5.075e-16,2.377e-16
32,16 × 16,16 × 8,0.007269,1.145395,0.01,5.075e-16,2.377e-16
32,16 × 16,32 × 8,0.007269,1.129931,0.01,5.075e-16,2.377e-16
32,32 × 32,32 × 8,0.007269,1.445237,0.01,5.075e-16,2.377e-16
32,16 × 16,16 × 16,0.007269,1.486884,0.00,5.075e-16,2.377e-16
32,32 × 32,16 × 16,0.007269,1.527129,0.00,5.075e-16,2.377e-16
32,16 × 16,32 × 16,0.007269,1.482803,0.00,5.075e-16,2.377e-16
32,32 × 32,32 × 16,0.007269,1.595316,0.00,5.075e-16,2.377e-16


## Лучшие конфигурации

Для каждого размера выбирается минимальное измеренное среднее время среди
всех десяти конфигураций. Это лучший результат данной серии на данном GPU;
пять повторов не позволяют считать небольшие различия устойчивым преимуществом.
Значение `CPU / GPU > 1` означает ускорение относительно последовательного CPU.


In [7]:
best_rows = []
for size in SIZES:
    rows = [r for r in gpu_measurements if int(r['n']) == size]
    if rows:
        best = min(rows, key=lambda r: float(r['gpu_ms']))
        best_rows.append(best)
if best_rows:
    show_table(['n', 'Лучшая сетка', 'Лучший блок', 'GPU, мс', 'CPU / GPU'],
               [[r['n'], f"{r['grid_x']} × {r['grid_y']}", f"{r['block_x']} × {r['block_y']}",
                 f"{float(r['gpu_ms']):.6f}", f"{float(r['cpu_ms']) / float(r['gpu_ms']):.2f}"]
                for r in best_rows])
else:
    print('Нет измеренных GPU-времён: лучшая конфигурация не выбрана.')
    print('Сетка 32 × 32 и блок 32 × 8 в lab4.cu — исходная настройка, а не измеренный оптимум.')


n,Лучшая сетка,Лучший блок,"GPU, мс",CPU / GPU
32,4 × 4,8 × 8,1.069679,0.01
256,16 × 16,16 × 8,9.012815,0.36
1024,16 × 16,32 × 8,119.924534,2.07


## Проверка CUDA-программы и пример из задания

Проверяются известные решения, выбор главного элемента, малые коэффициенты,
обход с шагом сетки и ошибочный ввод. После тестов программа решает пример:
$A=\begin{pmatrix}1&2\\3&4\end{pmatrix}$, $b=(5,6)^T$.
Ожидаемый ответ: $x=(-4,4.5)^T$.


In [8]:
if gpu_available:
    binary = work / 'lab4'
    if not binary.is_file() or binary.stat().st_mtime < (lab / 'lab4.cu').stat().st_mtime:
        if not nvcc:
            raise RuntimeError('Для сборки lab4.cu нужен установленный nvcc.')
        run([nvcc, '-O2', '-std=c++17', '-arch=sm_61', lab / 'lab4.cu', '-o', binary])
    checks = run([sys.executable, '-B', lab / 'tests' / 'check.py', binary, '--invalid'])
    print('Проверки CUDA-программы:')
    print(checks.stdout.strip())
    example = run([binary], input=(lab / 'data' / 'example.txt').read_text())
    assert [float(x) for x in example.stdout.split()] == [-4.0, 4.5]
    print('Ответ программы на пример из задания:')
    print(example.stdout.strip())
else:
    print('Проверки CUDA-программы требуют доступного GPU; текущие CSV остаются доступны.')


Проверки CUDA-программы:
Пройдено тестов: 21
Проверок ошибочного ввода: 12
Все тесты пройдены успешно.


Ответ программы на пример из задания:
-4.0000000000e+00 4.5000000000e+00


## Профилирование nvprof

Запуск из папки `lab4-variant3`:

```bash
python3 report/run_benchmarks.py --profile
```

Профиль сохраняется в `report/results/nvprof-256.log`. Ниже приведены время и число вызовов GPU-ядер и функций CUDA API.


In [9]:
profile_metadata = read_metadata('gpu')
profile_size = profile_metadata.get('arguments', {}).get('profile_size', PROFILE_SIZE)
profile_name = profile_metadata.get('profile_log', f'nvprof-{profile_size}.log')
profile_path = results / profile_name
profile_text = profile_path.read_text(errors='replace') if profile_path.is_file() else ''
failed_markers = ['run failed', 'profiler has not been started', 'no kernels were profiled',
                  '==error==', '\nerror:', 'no cuda-capable device']
profile_has_activity = bool(profile_text and 'GPU activities' in profile_text
                            and 'back_substitute' in profile_text
                            and (int(profile_size) == 1 or 'eliminate' in profile_text)
                            and not any(marker in profile_text.lower() for marker in failed_markers))

profile_rows = {'GPU activities': [], 'API calls': []}
pattern = re.compile(r'^\s*([\d.]+)%\s+([\d.]+(?:ns|us|ms|s))\s+(\d+)\s+'
                     r'([\d.]+(?:ns|us|ms|s))\s+\S+\s+\S+\s+(.+)$')

def milliseconds(value):
    match = re.fullmatch(r'([\d.]+)(ns|us|ms|s)', value)
    return float(match[1]) * {'ns': 1e-6, 'us': 1e-3, 'ms': 1, 's': 1000}[match[2]]

section = None
if profile_has_activity:
    for line in profile_text.splitlines():
        for label in profile_rows:
            if label + ':' in line:
                section = label
                line = line.split(label + ':', 1)[1]
                break
        match = pattern.fullmatch(line)
        if section and match:
            percent, total, calls, average, name = match.groups()
            name = 'Thrust: поиск главного элемента' if 'thrust' in name else name.split('(')[0]
            profile_rows[section].append({'name': name, 'percent': float(percent),
                                          'ms': milliseconds(total), 'calls': int(calls),
                                          'average_ms': milliseconds(average)})
    assert profile_rows['GPU activities'] and profile_rows['API calls'], 'Не удалось разобрать профиль.'
    print(f'Профиль решения n={profile_size}')
    for section, rows in profile_rows.items():
        print('\n' + section)
        show_table(['Операция', 'Доля, %', 'Всего, мс', 'Вызовов', 'Среднее, мкс'],
                   [[r['name'], f"{r['percent']:.2f}", f"{r['ms']:.6f}", r['calls'],
                     f"{r['average_ms'] * 1000:.3f}"] for r in rows])
    main_kernel = max(profile_rows['GPU activities'], key=lambda r: r['ms'])
    main_api = max(profile_rows['API calls'], key=lambda r: r['ms'])
    print(f"\nНаибольшая доля GPU-времени: {main_kernel['name']} — {main_kernel['percent']:.2f}%.")
    print(f"Наибольшая доля времени CUDA API: {main_api['name']} — {main_api['percent']:.2f}%.")
else:
    print('Полный профиль nvprof отсутствует. Успешные измерения времени решения системы можно анализировать отдельно.')
    if profile_text:
        print('\n'.join(profile_text.splitlines()[:20]))

Профиль решения n=256

GPU activities


Операция,"Доля, %","Всего, мс",Вызовов,"Среднее, мкс"
eliminate,66.91,4.653100,255,18.247
[CUDA memcpy DtoH],10.40,0.722870,512,1.411
Thrust: поиск главного элемента,8.04,0.558990,256,2.183
back_substitute,6.23,0.433390,256,1.692
compute_factors,5.65,0.392800,255,1.540
swap_rows,2.77,0.192680,128,1.505



API calls


Операция,"Доля, %","Всего, мс",Вызовов,"Среднее, мкс"
cudaLaunchKernel,30.34,5.046900,1150,4.388
cudaMemcpyAsync,19.07,3.171900,256,12.390
cudaMemcpy,18.74,3.117800,256,12.178
cudaStreamSynchronize,14.30,2.378300,512,4.645
cudaMalloc,6.03,1.002900,256,3.917
cudaFree,4.60,0.764390,256,2.985
cudaGetLastError,3.06,0.508890,6014,0.084
cudaGetDevice,2.07,0.343540,1280,0.268
cudaDeviceGetAttribute,1.40,0.233320,768,0.303
cudaPeekAtLastError,0.36,0.060624,512,0.118



Наибольшая доля GPU-времени: eliminate — 66.91%.
Наибольшая доля времени CUDA API: cudaLaunchKernel — 30.34%.


## Выводы по доступным измерениям

Численные выводы ниже строятся из проверенных CSV. При появлении измерений
GPU повторное выполнение notebook автоматически заполнит сравнение и выберет
лучшие конфигурации. Тесты используют хорошо обусловленные матрицы;
их результат не обобщается на произвольные плохо обусловленные СЛАУ.


In [10]:
max_cpu_error = max(float(r['cpu_rel_error']) for r in cpu_measurements)
max_cpu_residual = max(float(r['cpu_scaled_residual']) for r in cpu_measurements)
print(f'CPU: проверены размеры {", ".join(map(str, SIZES))}; '
      f'максимальная относительная ошибка {max_cpu_error:.3e}, невязка {max_cpu_residual:.3e}.')
if best_rows:
    for row in best_rows:
        speedup = float(row['cpu_ms']) / float(row['gpu_ms'])
        print(f"n={row['n']}: минимальное время GPU {float(row['gpu_ms']):.6f} мс; "
              f"сетка {row['grid_x']} × {row['grid_y']}, блок {row['block_x']} × {row['block_y']}; "
              f"CPU/GPU={speedup:.2f}.")
else:
    print('GPU: нет измерений, поэтому выводы об ускорении, лучшей сетке и корректности CUDA не сделаны.')
print('nvprof: ' + ('журнал содержит активность GPU при решении системы; результаты показаны выше.'
                    if profile_has_activity else 'профилирование остаётся невыполненным.'))


CPU: проверены размеры 32, 256, 1024; максимальная относительная ошибка 5.837e-15, невязка 2.737e-15.
n=32: минимальное время GPU 1.069679 мс; сетка 4 × 4, блок 8 × 8; CPU/GPU=0.01.
n=256: минимальное время GPU 9.012815 мс; сетка 16 × 16, блок 16 × 8; CPU/GPU=0.36.
n=1024: минимальное время GPU 119.924534 мс; сетка 16 × 16, блок 32 × 8; CPU/GPU=2.07.
nvprof: журнал содержит активность GPU при решении системы; результаты показаны выше.
